# Day 1-2 · Python Crash for LeetCode

这份 notebook 的目标是：在急投简历前，用最短时间恢复 Python 刷题手感，但不靠死记语法。

调整后的学习顺序是：

1. 先讲清楚底层原理：容器如何存数据、操作复杂度来自哪里、为什么某些写法慢。
2. 再用同一道小题分别给 Python 和 Go 实现，看出语言差异。
3. 最后做 2 个 Python 练习，用 `assert` 自测。

Go 不放练习题，只作为对照样例。重点不是让你同时刷两门语言，而是帮你把 Go 的直觉迁移成 Python 的正确写法。


In [1]:
from collections import defaultdict, Counter, deque
from functools import lru_cache
from itertools import combinations, permutations, accumulate
import heapq
import bisect
import asyncio
import time


## 1. List / Go Slice：动态数组

### 底层原理先讲清楚

Python `list` 和 Go `slice` 都经常被叫作“数组”，但底层模型不同。

Python `list`：

- 本质是动态数组，数组里存的是**对象引用**，不是对象本体。
- `append` 均摊 `O(1)`：容量不够时会申请更大的连续空间，并把引用搬过去。
- `a[i]` 是 `O(1)`，因为能通过下标直接算出引用位置。
- `insert(0, x)` / `pop(0)` 是 `O(n)`，因为头部变动会导致后面元素整体移动。
- 切片 `a[i:j:k]` 会创建新 list，时间和空间都和切片长度相关。
- `sorted(a)` 返回新 list；`a.sort()` 原地排序并返回 `None`。

Go slice：

- slice 是一个三元描述符：指向底层数组的指针、长度 `len`、容量 `cap`。
- `append` 可能复用原底层数组，也可能扩容并拷贝到新数组。
- 切片表达式通常共享底层数组；修改子 slice 可能影响原 slice。
- Go 的 slice 存的是具体类型元素，内存布局通常比 Python list 更紧凑。

刷题里的核心直觉：

- Python list 适合做顺序扫描、答案收集、栈、排序后的线性处理。
- 不适合频繁队头删除；队列用 `deque`。
- Python 切片不是 Go slice 视图，不要以为它零成本。

### 同题对照：平方后排序

题意：给一个整数数组，返回每个数平方后的升序结果。

重点：看 list comprehension / `sorted` 与 Go 显式循环的差异。


In [2]:
def sorted_squares_python(nums):
    return sorted(x * x for x in nums)

sample_nums = [-4, -1, 0, 3, 10]
sample_output = sorted_squares_python(sample_nums)
print("input:", sample_nums)
print("output:", sample_output)

assert sample_output == [0, 1, 9, 16, 100]


input: [-4, -1, 0, 3, 10]
output: [0, 1, 9, 16, 100]


```go
package main

import "sort"

func sortedSquaresGo(nums []int) []int {
    ans := make([]int, 0, len(nums))
    for _, x := range nums {
        ans = append(ans, x*x)
    }
    sort.Ints(ans)
    return ans
}
```

差异要点：Python 更强调“把变换表达出来”；Go 更强调显式分配、循环和类型。

### 练习 A：移动零

思路精髓：原地数组题常用“读指针发现元素，写指针压缩有效元素”。


In [21]:
def move_zeroes1(nums):
    # TODO: 原地把 0 移到末尾，保持非零元素相对顺序
    # 定义一个指针，用于记录非0元素的位置
    write = 0
    for x in nums:
        # 先将非0元素移到前面
        if x != 0:
            nums[write] = x
            write += 1
    # 从 write 位置开始，到数组末尾，填充0
    for i in range(write, len(nums)):
        nums[i] = 0
    return nums

    
arr = [0, 1, 0, 3, 12]
move_zeroes1(arr)
print(arr)
assert arr == [1, 3, 12, 0, 0]


[1, 3, 12, 0, 0]


参考实现：


In [19]:
def move_zeroes(nums):
    write = 0
    for x in nums:
        if x != 0:
            nums[write] = x
            write += 1
            print(nums)
    for i in range(write, len(nums)):
        nums[i] = 0
        # print(nums)

sample_nums = [0, 1, 0, 3, 12]
print("input:", sample_nums)
move_zeroes(sample_nums)
print("output:", sample_nums)

assert sample_nums == [1, 3, 12, 0, 0]


input: [0, 1, 0, 3, 12]
[1, 1, 0, 3, 12]
[1, 3, 0, 3, 12]
[1, 3, 12, 3, 12]
output: [1, 3, 12, 0, 0]


### 练习 B：合并区间

思路精髓：排序把“可能重叠”变成相邻关系，然后只维护最后一个合并区间。


In [2]:
def merge_intervals(intervals):
    if not intervals:
        return []

    intervals.sort(key=lambda x: x[0])

    ans = []
    # 遍历每个区间
    for start, end in intervals:
        # 如果当前区间和上一个区间不重叠，直接加入结果
        if not ans or start > ans[-1][1]:
            ans.append([start, end])
        else:
            # 如果当前区间和上一个区间重叠，更新最后一个区间的结束位置, 取最大
            ans[-1][1] = max(ans[-1][1], end)

    return ans

intervals=[[1,3],[2,6],[8,10],[15,18]]
output=merge_intervals(intervals)
print(output)

assert merge_intervals([[1,3],[2,6],[8,10],[15,18]]) == [[1,6],[8,10],[15,18]]
assert merge_intervals([[1,4],[4,5]]) == [[1,5]]


[[1, 6], [8, 10], [15, 18]]


参考实现：


In [ ]:
def merge_intervals(intervals):
    intervals = sorted(intervals, key=lambda x: x[0])
    ans = []
    for start, end in intervals:
        if not ans or start > ans[-1][1]:
            ans.append([start, end])
        else:
            ans[-1][1] = max(ans[-1][1], end)
    return ans

sample_intervals = [[1,3],[2,6],[8,10],[15,18]]
sample_output = merge_intervals(sample_intervals)
print("input:", sample_intervals)
print("output:", sample_output)

assert sample_output == [[1,6],[8,10],[15,18]]
assert merge_intervals([[1,4],[4,5]]) == [[1,5]]


## 2. Dict / Go Map：哈希表

### 底层原理先讲清楚

哈希表解决的问题是：把 key 映射到数组位置，从而把查找从线性扫描降到平均 `O(1)`。

共同原理：

- 对 key 计算 hash 值。
- hash 值定位到某个桶或槽位。
- 发生冲突时，用开放寻址、链表、溢出桶等策略解决。
- 当装载因子太高，哈希表会扩容并迁移数据。

Python `dict`：

- 平均查找、插入、删除是 `O(1)`，最坏情况可能退化，但日常刷题按平均复杂度分析。
- key 必须可哈希：`int`、`str`、`tuple` 可以；`list` 不可以。
- Python 3.7+ 保留插入顺序，但刷算法题时不要把“有序 dict”当核心假设，除非题目需要。
- `dict.get(k, default)`、`defaultdict`、`Counter` 是刷题高频工具。

Go `map`：

- `map[K]V` 的 K 必须是可比较类型；slice、map、func 不能做 key。
- 读取不存在的 key 会返回 value 类型的零值，所以常用 `v, ok := m[k]` 区分“不存在”和“存在但值是零”。
- nil map 可以读，不能写；写之前要 `make(map[K]V)`。

刷题里的核心直觉：

- 需要“过去是否出现过”：dict/map。
- 需要“值到下标”：dict/map。
- 需要“计数”：Python 优先 `Counter`，Go 手写 `map[T]int`。
- 需要“分组”：Python `defaultdict(list)`，Go 手动初始化 slice。

### 同题对照：Two Sum


In [ ]:
def two_sum_python(nums, target):
    seen = {}
    for i, x in enumerate(nums):
        need = target - x
        if need in seen:
            return [seen[need], i]
        seen[x] = i
    return []

sample_nums = [2, 7, 11, 15]
sample_target = 9
sample_output = two_sum_python(sample_nums, sample_target)
print("input:", sample_nums, "target:", sample_target)
print("output:", sample_output)

assert sample_output == [0, 1]


```go
func twoSumGo(nums []int, target int) []int {
    seen := make(map[int]int)
    for i, x := range nums {
        need := target - x
        if j, ok := seen[need]; ok {
            return []int{j, i}
        }
        seen[x] = i
    }
    return nil
}
```

差异要点：Python 用 `in` 判断 key；Go 用双返回值 `value, ok`。

### 练习 A：有效异位词

思路精髓：两个字符串是否异位，本质是字符频次向量是否相同。


In [ ]:
def is_anagram(s, t):
    # TODO
    pass

assert is_anagram("anagram", "nagaram") is True
assert is_anagram("rat", "car") is False


参考实现：


In [ ]:
def is_anagram(s, t):
    return Counter(s) == Counter(t)

sample_s = "anagram"
sample_t = "nagaram"
sample_output = is_anagram(sample_s, sample_t)
print("input:", sample_s, sample_t)
print("output:", sample_output)

assert sample_output is True
assert is_anagram("rat", "car") is False


### 练习 B：字母异位词分组

思路精髓：分组题的关键是设计规范 key，把不同表面形式映射成同一个 key。


In [ ]:
def group_anagrams(strs):
    # TODO
    pass

ans = group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
normalized = sorted(sorted(group) for group in ans)
assert normalized == sorted([sorted(["eat", "tea", "ate"]), sorted(["tan", "nat"]), ["bat"]])


参考实现：


In [ ]:
def group_anagrams(strs):
    groups = defaultdict(list)
    for s in strs:
        key = ''.join(sorted(s))
        groups[key].append(s)
    return list(groups.values())

sample_strs = ["eat", "tea", "tan", "ate", "nat", "bat"]
sample_output = group_anagrams(sample_strs)
print("input:", sample_strs)
print("output:", sample_output)

normalized = sorted(sorted(group) for group in sample_output)
assert normalized == sorted([sorted(["eat", "tea", "ate"]), sorted(["tan", "nat"]), ["bat"]])


## 3. Set / Go Map-as-Set：存在性集合

### 底层原理先讲清楚

Set 是“只存 key、不关心 value”的哈希表。它回答的问题是：某个元素是否存在。

Python `set`：

- 平均 `add`、`discard`、`in` 都是 `O(1)`。
- 空集合是 `set()`，不是 `{}`；`{}` 是 dict。
- `discard(x)` 不存在也不报错，`remove(x)` 不存在会报错。
- 支持集合运算：`a & b` 交集、`a | b` 并集、`a - b` 差集。

Go 没有内置 set：

- 常用 `map[T]struct{}` 表达 set，因为 `struct{}` 不占额外存储空间。
- 也可以用 `map[T]bool`，但语义和空间都略差。

刷题里的核心直觉：

- 如果只问“有没有”，用 set。
- 如果要次数，用 dict/Counter。
- 如果要位置，用 dict。

### 同题对照：存在重复元素


In [ ]:
def contains_duplicate_python(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False

sample_nums = [1, 2, 3, 1]
sample_output = contains_duplicate_python(sample_nums)
print("input:", sample_nums)
print("output:", sample_output)

assert sample_output is True
assert contains_duplicate_python([1, 2, 3, 4]) is False


```go
func containsDuplicateGo(nums []int) bool {
    seen := make(map[int]struct{})
    for _, x := range nums {
        if _, ok := seen[x]; ok {
            return true
        }
        seen[x] = struct{}{}
    }
    return false
}
```

差异要点：Python 有专门的 `set`；Go 需要用 map 模拟。

### 练习 A：两个数组的唯一交集

思路精髓：集合交集直接表达“同时存在”。


In [ ]:
def unique_intersection(a, b):
    # TODO
    pass

assert sorted(unique_intersection([1,2,2,1], [2,2])) == [2]
assert sorted(unique_intersection([4,9,5], [9,4,9,8,4])) == [4,9]


参考实现：


In [ ]:
def unique_intersection(a, b):
    return list(set(a) & set(b))

sample_a = [4, 9, 5]
sample_b = [9, 4, 9, 8, 4]
sample_output = sorted(unique_intersection(sample_a, sample_b))
print("input:", sample_a, sample_b)
print("output:", sample_output)

assert sorted(unique_intersection([1,2,2,1], [2,2])) == [2]
assert sample_output == [4,9]


### 练习 B：最长无重复子串

思路精髓：滑动窗口里的 set 表示当前窗口有哪些字符；每个字符最多进出一次，所以 `O(n)`。


In [ ]:
def length_of_longest_substring(s):
    # TODO
    pass

assert length_of_longest_substring("abcabcbb") == 3
assert length_of_longest_substring("bbbbb") == 1
assert length_of_longest_substring("") == 0


参考实现：


In [ ]:
def length_of_longest_substring(s):
    seen = set()
    left = 0
    best = 0
    for right, ch in enumerate(s):
        while ch in seen:
            seen.discard(s[left])
            left += 1
        seen.add(ch)
        best = max(best, right - left + 1)
    return best

sample_s = "abcabcbb"
sample_output = length_of_longest_substring(sample_s)
print("input:", sample_s)
print("output:", sample_output)

assert sample_output == 3
assert length_of_longest_substring("bbbbb") == 1
assert length_of_longest_substring("") == 0


## 4. String：不可变字符序列 vs 字节序列

### 底层原理先讲清楚

Python `str`：

- 字符串不可变；修改字符串会创建新对象。
- 循环里反复 `ans += part` 可能导致大量中间字符串，构造结果时优先 `parts.append(...)` 后 `''.join(parts)`。
- 迭代 `str` 得到的是 Unicode 字符。
- 英文字母题可以用 `ord(ch) - ord('a')` 映射到 `0..25`。

Go `string`：

- 本质是只读字节序列，通常存 UTF-8。
- `len(s)` 是字节数，不是字符数。
- `for i, r := range s` 中 `i` 是字节下标，`r` 是 rune。
- 构造字符串常用 `strings.Builder` 或 `[]byte`。

刷题里的核心直觉：

- 只处理小写英文字母时，Python 和 Go 都可以按字符/byte 做桶计数。
- 涉及 Unicode 时，Go 的 byte/rune 区分很重要。
- Python 字符串题常用 `split`、`join`、`strip` 把解析和构造写短。

### 同题对照：反转句子中的单词


In [ ]:
def reverse_words_python(s):
    return " ".join(reversed(s.split()))

sample_s = "  hello world  "
sample_output = reverse_words_python(sample_s)
print("input:", repr(sample_s))
print("output:", sample_output)

assert sample_output == "world hello"


```go
import "strings"

func reverseWordsGo(s string) string {
    words := strings.Fields(s)
    for i, j := 0, len(words)-1; i < j; i, j = i+1, j-1 {
        words[i], words[j] = words[j], words[i]
    }
    return strings.Join(words, " ")
}
```

差异要点：Python 的 `split()` 默认已经完成 trim 和合并空白；Go 对应是 `strings.Fields`。

### 练习 A：有效回文，只看字母数字

思路精髓：双指针跳过无关字符，比较规范化后的字符。


In [ ]:
def is_palindrome_text(s):
    # TODO
    pass

assert is_palindrome_text("A man, a plan, a canal: Panama") is True
assert is_palindrome_text("race a car") is False


参考实现：


In [ ]:
def is_palindrome_text(s):
    left, right = 0, len(s) - 1
    while left < right:
        while left < right and not s[left].isalnum():
            left += 1
        while left < right and not s[right].isalnum():
            right -= 1
        if s[left].lower() != s[right].lower():
            return False
        left += 1
        right -= 1
    return True

sample_s = "A man, a plan, a canal: Panama"
sample_output = is_palindrome_text(sample_s)
print("input:", sample_s)
print("output:", sample_output)

assert sample_output is True
assert is_palindrome_text("race a car") is False


### 练习 B：字母频次签名

思路精髓：把字符串压缩成可哈希特征，可用于分组或比较。


In [ ]:
def letter_signature(s):
    # TODO
    pass

assert letter_signature("abbc")[:4] == (1, 2, 1, 0)
assert letter_signature("eat") == letter_signature("tea")


参考实现：


In [ ]:
def letter_signature(s):
    freq = [0] * 26
    for ch in s:
        freq[ord(ch) - ord('a')] += 1
    return tuple(freq)

sample_s = "abbc"
sample_output = letter_signature(sample_s)
print("input:", sample_s)
print("output first 4:", sample_output[:4])

assert sample_output[:4] == (1, 2, 1, 0)
assert letter_signature("eat") == letter_signature("tea")


## 5. Stack / Queue：访问顺序决定结构

### 底层原理先讲清楚

栈和队列不是“某个语言类型”，而是两种访问纪律。

Stack：后进先出。

- Python 用 list 尾部 `append/pop`，都是均摊 `O(1)`。
- Go 通常用 slice 尾部 `append` 和 `stack = stack[:len(stack)-1]`。

Queue：先进先出。

- Python 不要用 list 的 `pop(0)`，这是 `O(n)`。
- Python 用 `collections.deque`，两端入队出队都是 `O(1)`。
- Go 可以用 slice 加 head 指针，避免每次从头删除导致搬移。

刷题里的核心直觉：

- 括号匹配、单调栈、DFS 回退：stack。
- BFS、层序遍历、无权图最短路：queue。

### 同题对照：有效括号


In [ ]:
def is_valid_parentheses_python(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = []
    for ch in s:
        if ch in pairs.values():
            stack.append(ch)
        else:
            if not stack or stack.pop() != pairs[ch]:
                return False
    return not stack

sample_s = "()[]{}"
sample_output = is_valid_parentheses_python(sample_s)
print("input:", sample_s)
print("output:", sample_output)

assert sample_output is True
assert is_valid_parentheses_python("([)]") is False


```go
func isValidParenthesesGo(s string) bool {
    pairs := map[byte]byte{')': '(', ']': '[', '}': '{'}
    stack := make([]byte, 0, len(s))
    for i := 0; i < len(s); i++ {
        ch := s[i]
        if ch == '(' || ch == '[' || ch == '{' {
            stack = append(stack, ch)
        } else {
            if len(stack) == 0 || stack[len(stack)-1] != pairs[ch] {
                return false
            }
            stack = stack[:len(stack)-1]
        }
    }
    return len(stack) == 0
}
```

差异要点：Python list 当 stack 很自然；Go slice 也能当 stack，但弹出要手动截短。

### 练习 A：用栈去掉相邻重复字符

思路精髓：当前字符如果和栈顶相同，就抵消；否则入栈。


In [ ]:
def remove_adjacent_duplicates(s):
    # TODO
    pass

assert remove_adjacent_duplicates("abbaca") == "ca"
assert remove_adjacent_duplicates("azxxzy") == "ay"


参考实现：


In [ ]:
def remove_adjacent_duplicates(s):
    stack = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()
        else:
            stack.append(ch)
    return ''.join(stack)

sample_s = "abbaca"
sample_output = remove_adjacent_duplicates(sample_s)
print("input:", sample_s)
print("output:", sample_output)

assert sample_output == "ca"
assert remove_adjacent_duplicates("azxxzy") == "ay"


### 练习 B：网格最短路 BFS

思路精髓：无权图最短路用 BFS。第一次到达终点时，距离一定最短。


In [ ]:
def shortest_path_grid(grid):
    # TODO
    pass

assert shortest_path_grid([[0,0,0],[1,1,0],[0,0,0]]) == 4
assert shortest_path_grid([[0,1],[1,0]]) == -1


参考实现：


In [ ]:
def shortest_path_grid(grid):
    m, n = len(grid), len(grid[0])
    if grid[0][0] == 1 or grid[m - 1][n - 1] == 1:
        return -1
    q = deque([(0, 0, 0)])
    seen = {(0, 0)}
    for_dirs = [(1, 0), (-1, 0), (0, 1), (0, -1)]
    while q:
        r, c, dist = q.popleft()
        if (r, c) == (m - 1, n - 1):
            return dist
        for dr, dc in for_dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == 0 and (nr, nc) not in seen:
                seen.add((nr, nc))
                q.append((nr, nc, dist + 1))
    return -1

sample_grid = [[0,0,0],[1,1,0],[0,0,0]]
sample_output = shortest_path_grid(sample_grid)
print("input:", sample_grid)
print("output:", sample_output)

assert sample_output == 4
assert shortest_path_grid([[0,1],[1,0]]) == -1


## 6. Heap：用数组表示的完全二叉树

### 底层原理先讲清楚

堆是一棵完全二叉树，通常用数组存。

下标关系：

- parent: `(i - 1) // 2`
- left child: `2 * i + 1`
- right child: `2 * i + 2`

最小堆性质：每个父节点都不大于子节点，所以根节点一定是全局最小值。但堆不是排序数组，除了根之外，其他位置没有全局顺序保证。

复杂度：

- `push`：把元素放到末尾，再向上调整，`O(log n)`。
- `pop`：把根弹出，用末尾元素补根，再向下调整，`O(log n)`。
- `heapify`：从最后一个非叶子节点向前调整，整体 `O(n)`。

Python `heapq`：

- 标准库只提供最小堆。
- 最大堆常用负数 trick：存 `-x`。
- 元素可以是 tuple，按字典序比较，常用于 `(priority, data)`。

Go `container/heap`：

- 需要实现 `heap.Interface`：`Len/Less/Swap/Push/Pop`。
- 代码比 Python 长，但类型控制更明确。

刷题里的核心直觉：

- 每次要拿当前最小/最大，用堆。
- 只关心 Top K，用大小为 K 的堆。
- 合并 K 路有序序列，用堆维护每一路当前头部。

### 同题对照：第 K 大元素


In [ ]:
def kth_largest_python(nums, k):
    heap = []
    for x in nums:
        heapq.heappush(heap, x)
        if len(heap) > k:
            heapq.heappop(heap)
    return heap[0]

sample_nums = [3,2,1,5,6,4]
sample_k = 2
sample_output = kth_largest_python(sample_nums, sample_k)
print("input:", sample_nums, "k:", sample_k)
print("output:", sample_output)

assert sample_output == 5


```go
import "container/heap"

type MinHeap []int

func (h MinHeap) Len() int           { return len(h) }
func (h MinHeap) Less(i, j int) bool { return h[i] < h[j] }
func (h MinHeap) Swap(i, j int)      { h[i], h[j] = h[j], h[i] }
func (h *MinHeap) Push(x any)        { *h = append(*h, x.(int)) }
func (h *MinHeap) Pop() any {
    old := *h
    x := old[len(old)-1]
    *h = old[:len(old)-1]
    return x
}

func kthLargestGo(nums []int, k int) int {
    h := &MinHeap{}
    heap.Init(h)
    for _, x := range nums {
        heap.Push(h, x)
        if h.Len() > k {
            heap.Pop(h)
        }
    }
    return (*h)[0]
}
```

差异要点：Python heap 写法短很多；Go 的堆需要接口样板，但可定制性强。

### 练习 A：Top K 高频元素

思路精髓：先计数，再用大小为 K 的最小堆保留频率最高的 K 个候选。


In [ ]:
def top_k_frequent(nums, k):
    # TODO
    pass

assert sorted(top_k_frequent([1,1,1,2,2,3], 2)) == [1,2]
assert top_k_frequent([1], 1) == [1]


参考实现：


In [ ]:
def top_k_frequent(nums, k):
    count = Counter(nums)
    heap = []
    for x, freq in count.items():
        heapq.heappush(heap, (freq, x))
        if len(heap) > k:
            heapq.heappop(heap)
    return [x for freq, x in heap]

sample_nums = [1,1,1,2,2,3]
sample_k = 2
sample_output = sorted(top_k_frequent(sample_nums, sample_k))
print("input:", sample_nums, "k:", sample_k)
print("output:", sample_output)

assert sample_output == [1,2]
assert top_k_frequent([1], 1) == [1]


### 练习 B：合并 K 个有序数组

思路精髓：堆里只放每个数组当前暴露的最小候选，弹出一个再补同数组下一个。


In [ ]:
def merge_k_sorted_arrays(arrays):
    # TODO
    pass

assert merge_k_sorted_arrays([[1,4,5],[1,3,4],[2,6]]) == [1,1,2,3,4,4,5,6]
assert merge_k_sorted_arrays([[], [2], [1, 3]]) == [1,2,3]


参考实现：


In [ ]:
def merge_k_sorted_arrays(arrays):
    heap = []
    for ai, arr in enumerate(arrays):
        if arr:
            heapq.heappush(heap, (arr[0], ai, 0))
    ans = []
    while heap:
        val, ai, ei = heapq.heappop(heap)
        ans.append(val)
        ni = ei + 1
        if ni < len(arrays[ai]):
            heapq.heappush(heap, (arrays[ai][ni], ai, ni))
    return ans

sample_arrays = [[1,4,5],[1,3,4],[2,6]]
sample_output = merge_k_sorted_arrays(sample_arrays)
print("input:", sample_arrays)
print("output:", sample_output)

assert sample_output == [1,1,2,3,4,4,5,6]
assert merge_k_sorted_arrays([[], [2], [1, 3]]) == [1,2,3]


## 7. Bisect / Binary Search：边界查找

### 底层原理先讲清楚

二分查找的本质不是“找某个值”，而是“在单调条件上找边界”。

对于有序数组：

- lower_bound：第一个 `>= target` 的位置。
- upper_bound：第一个 `> target` 的位置。

Python `bisect`：

- `bisect_left(a, x)`：lower_bound。
- `bisect_right(a, x)`：upper_bound。
- 查找位置 `O(log n)`。
- 如果真的插入 list，中间元素要移动，插入是 `O(n)`。

Go `sort.Search`：

- 用 predicate 表达“从哪里开始条件为 true”。
- 常见写法：`sort.Search(len(nums), func(i int) bool { return nums[i] >= target })`。

刷题里的核心直觉：

- 有序数组找边界，先想 bisect。
- 统计 `[lo, hi]` 个数：`right(hi) - left(lo)`。
- 不要把二分写成“猜 mid”，要写成“维护答案边界”。

### 同题对照：搜索插入位置


In [ ]:
def search_insert_python(nums, target):
    return bisect.bisect_left(nums, target)

sample_nums = [1,3,5,6]
sample_target = 2
sample_output = search_insert_python(sample_nums, sample_target)
print("input:", sample_nums, "target:", sample_target)
print("output:", sample_output)

assert search_insert_python([1,3,5,6], 5) == 2
assert sample_output == 1


```go
import "sort"

func searchInsertGo(nums []int, target int) int {
    return sort.Search(len(nums), func(i int) bool {
        return nums[i] >= target
    })
}
```

差异要点：Python 直接给 lower/upper bound；Go 用 predicate 更通用。

### 练习 A：统计 target 出现次数

思路精髓：`right - left` 就是等于 target 的元素个数。


In [ ]:
def count_equal(nums, target):
    # TODO
    pass

assert count_equal([1,2,2,2,3], 2) == 3
assert count_equal([1,3,5], 2) == 0


参考实现：


In [ ]:
def count_equal(nums, target):
    return bisect.bisect_right(nums, target) - bisect.bisect_left(nums, target)

sample_nums = [1,2,2,2,3]
sample_target = 2
sample_output = count_equal(sample_nums, sample_target)
print("input:", sample_nums, "target:", sample_target)
print("output:", sample_output)

assert sample_output == 3
assert count_equal([1,3,5], 2) == 0


### 练习 B：统计闭区间内元素个数

思路精髓：小于等于 `hi` 的数量减去小于 `lo` 的数量。


In [ ]:
def count_in_range(nums, lo, hi):
    # TODO
    pass

assert count_in_range([1,2,2,3,5,8], 2, 5) == 4
assert count_in_range([1,2,2,3,5,8], 4, 7) == 1


参考实现：


In [ ]:
def count_in_range(nums, lo, hi):
    return bisect.bisect_right(nums, hi) - bisect.bisect_left(nums, lo)

sample_nums = [1,2,2,3,5,8]
sample_lo = 2
sample_hi = 5
sample_output = count_in_range(sample_nums, sample_lo, sample_hi)
print("input:", sample_nums, "range:", [sample_lo, sample_hi])
print("output:", sample_output)

assert sample_output == 4
assert count_in_range([1,2,2,3,5,8], 4, 7) == 1


## 8. Itertools：迭代器和枚举空间

### 底层原理先讲清楚

`itertools` 不是新的数据结构，而是一组高效的迭代器工具。它的关键是“惰性”：很多结果不会一次性全部生成，而是在迭代时逐个产生。

Python：

- `combinations(nums, k)`：组合，不考虑顺序。
- `permutations(nums, k)`：排列，考虑顺序。
- `accumulate(nums)`：前缀累计。
- 返回的通常是 iterator；被消费后不能自动重用。

Go：

- 标准库没有等价的组合/排列生成器，通常手写 DFS 或循环。
- 前缀和通常手写数组。

复杂度直觉：

- 工具只能减少代码量，不能改变组合爆炸。
- `C(n, k)`、`P(n, k)` 的规模仍然可能很大。

刷题里的核心直觉：

- 规模小、明确要枚举组合/排列时，用 itertools 提速写题。
- 面试里要能解释枚举空间有多大。

### 同题对照：所有二元组合和


In [ ]:
def pair_sums_python(nums):
    return [a + b for a, b in combinations(nums, 2)]

sample_nums = [1, 2, 3]
sample_output = sorted(pair_sums_python(sample_nums))
print("input:", sample_nums)
print("output:", sample_output)

assert sample_output == [3, 4, 5]


```go
func pairSumsGo(nums []int) []int {
    ans := make([]int, 0)
    for i := 0; i < len(nums); i++ {
        for j := i + 1; j < len(nums); j++ {
            ans = append(ans, nums[i]+nums[j])
        }
    }
    return ans
}
```

差异要点：Python 可以把“组合”作为库概念直接表达；Go 通常显式写枚举边界。

### 练习 A：生成所有长度为 2 的组合

思路精髓：组合不关心顺序，所以 `(1,2)` 和 `(2,1)` 只出现一次。


In [ ]:
def all_pairs(nums):
    # TODO
    pass

assert all_pairs([1,2,3]) == [(1,2), (1,3), (2,3)]
assert all_pairs([1]) == []


参考实现：


In [ ]:
def all_pairs(nums):
    return list(combinations(nums, 2))

sample_nums = [1,2,3]
sample_output = all_pairs(sample_nums)
print("input:", sample_nums)
print("output:", sample_output)

assert sample_output == [(1,2), (1,3), (2,3)]
assert all_pairs([1]) == []


### 练习 B：前缀和区间查询

思路精髓：多放一个开头的 0，区间和 `sum(l..r) = prefix[r+1] - prefix[l]`。


In [ ]:
def build_prefix(nums):
    # TODO
    pass

def range_sum(prefix, l, r):
    # TODO
    pass

prefix = build_prefix([2, 4, 6, 8])
assert prefix == [0, 2, 6, 12, 20]
assert range_sum(prefix, 1, 3) == 18


参考实现：


In [ ]:
def build_prefix(nums):
    return [0] + list(accumulate(nums))

def range_sum(prefix, l, r):
    return prefix[r + 1] - prefix[l]

sample_nums = [2, 4, 6, 8]
prefix = build_prefix(sample_nums)
sample_output = range_sum(prefix, 1, 3)
print("input:", sample_nums)
print("prefix:", prefix)
print("range_sum(1, 3):", sample_output)

assert prefix == [0, 2, 6, 12, 20]
assert sample_output == 18


## 9. @lru_cache / Memo Map：记忆化搜索

### 底层原理先讲清楚

记忆化搜索解决的是“递归状态重复计算”。

无缓存递归的问题：

- 递归树里相同子问题会出现很多次。
- 例如 Fibonacci 的 `f(5)` 会重复算 `f(3)`、`f(2)`。

加缓存后的模型：

- 把 `函数参数 -> 返回值` 存进哈希表。
- 同一个状态第二次出现时直接返回。
- 复杂度从“递归树规模”降到“状态数 × 每个状态转移代价”。

Python `@lru_cache(None)`：

- 自动用函数参数作为 key。
- 参数必须可哈希；list 要转 tuple。
- 适合 LeetCode 里快速写 top-down DP。

Go：

- 通常手写 `memo := map[State]Value{}`。
- 状态如果复杂，需要定义 struct 作为 key，字段必须可比较。

刷题里的核心直觉：

- 先定义状态 `dp(...)` 的含义。
- 再写 base case。
- 最后写状态转移。
- 如果状态会重复，缓存就是必要的。

### 同题对照：爬楼梯


In [ ]:
def climb_stairs_python(n):
    @lru_cache(None)
    def dp(i):
        if i <= 2:
            return i
        return dp(i - 1) + dp(i - 2)
    return dp(n)

sample_n = 5
sample_output = climb_stairs_python(sample_n)
print("input:", sample_n)
print("output:", sample_output)

assert sample_output == 8


```go
func climbStairsGo(n int) int {
    memo := make(map[int]int)
    var dp func(int) int
    dp = func(i int) int {
        if i <= 2 {
            return i
        }
        if v, ok := memo[i]; ok {
            return v
        }
        memo[i] = dp(i-1) + dp(i-2)
        return memo[i]
    }
    return dp(n)
}
```

差异要点：Python 装饰器隐藏 memo map；Go 把 memo 显式写出来。

### 练习 A：最小花费爬楼梯

思路精髓：先定义 `dp(i)` 是到达第 i 阶的最小花费，楼顶 `n` 不需要付费。


In [ ]:
def min_cost_climbing_stairs(cost):
    # TODO
    pass

assert min_cost_climbing_stairs([10, 15, 20]) == 15
assert min_cost_climbing_stairs([1,100,1,1,1,100,1,1,100,1]) == 6


参考实现：


In [ ]:
def min_cost_climbing_stairs(cost):
    n = len(cost)

    @lru_cache(None)
    def dp(i):
        if i <= 1:
            return 0
        return min(dp(i - 1) + cost[i - 1], dp(i - 2) + cost[i - 2])

    return dp(n)

sample_cost = [10, 15, 20]
sample_output = min_cost_climbing_stairs(sample_cost)
print("input:", sample_cost)
print("output:", sample_output)

assert sample_output == 15
assert min_cost_climbing_stairs([1,100,1,1,1,100,1,1,100,1]) == 6


### 练习 B：单词拆分

思路精髓：`dp(i)` 表示 `s[i:]` 能否被词典拆分。状态转移枚举从 i 开始的下一个单词。


In [ ]:
def word_break(s, word_dict):
    # TODO
    pass

assert word_break("leetcode", ["leet", "code"]) is True
assert word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]) is False


参考实现：


In [ ]:
def word_break(s, word_dict):
    words = set(word_dict)

    @lru_cache(None)
    def dp(i):
        if i == len(s):
            return True
        for j in range(i + 1, len(s) + 1):
            if s[i:j] in words and dp(j):
                return True
        return False

    return dp(0)

sample_s = "leetcode"
sample_words = ["leet", "code"]
sample_output = word_break(sample_s, sample_words)
print("input:", sample_s, sample_words)
print("output:", sample_output)

assert sample_output is True
assert word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]) is False


## 10. Python Async / Go Goroutine：并发模型

### 底层原理先讲清楚

这节不是 LeetCode 高频内容，但对后端、Agent 工程、平台岗位很有用。

Python `asyncio`：

- `async def` 调用后得到 coroutine，不会自动跑完整函数。
- `await` 表示当前 coroutine 主动让出控制权。
- event loop 负责在多个 coroutine 之间调度。
- `asyncio.create_task(coro)` 把 coroutine 注册成任务。
- `asyncio.gather(*tasks)` 等多个任务完成，并按传入顺序返回结果。
- 适合 I/O 密集，不适合 CPU 密集。

Go goroutine：

- `go f()` 会启动 goroutine，由 Go runtime 调度。
- goroutine 可以并发执行，channel 常用于通信和同步。
- Go 的并发模型更接近“启动轻量线程”；Python asyncio 更接近“单线程事件循环中的协作式调度”。

关键区别：

- Python async 里的任务只有遇到 `await` 才让出控制权。
- 如果在 async 函数里写 `time.sleep()`，会阻塞整个 event loop。
- Go 里阻塞一个 goroutine 不等于阻塞所有 goroutine，runtime 会调度其他 goroutine。

### 同题对照：并发执行三个请求


In [ ]:
async def fake_fetch(name, delay):
    await asyncio.sleep(delay)
    return f"{name}: done"

async def fetch_three_python():
    requests = [("a", 0.1), ("b", 0.2), ("c", 0.05)]
    tasks = [asyncio.create_task(fake_fetch(name, delay)) for name, delay in requests]
    output = await asyncio.gather(*tasks)
    print("input:", requests)
    print("output:", output)
    return output

assert await fetch_three_python() == ["a: done", "b: done", "c: done"]


```go
import (
    "fmt"
    "sync"
    "time"
)

func fakeFetchGo(name string, delay time.Duration) string {
    time.Sleep(delay)
    return fmt.Sprintf("%s: done", name)
}

func fetchThreeGo() []string {
    names := []string{"a", "b", "c"}
    delays := []time.Duration{100 * time.Millisecond, 200 * time.Millisecond, 50 * time.Millisecond}
    results := make([]string, len(names))

    var wg sync.WaitGroup
    for i := range names {
        wg.Add(1)
        go func(i int) {
            defer wg.Done()
            results[i] = fakeFetchGo(names[i], delays[i])
        }(i)
    }
    wg.Wait()
    return results
}
```

差异要点：Python 用 event loop + await；Go 用 goroutine + WaitGroup。两者都能保持结果顺序，只要把结果写回对应下标。

### 练习 A：并发模拟多个请求

思路精髓：先创建所有 task，再 gather；总耗时接近最大 delay，而不是 delay 之和。


In [ ]:
async def fetch_all(requests):
    # TODO: requests 是 [(name, delay), ...]
    pass

start = time.perf_counter()
results = await fetch_all([("a", 0.1), ("b", 0.2), ("c", 0.05)])
elapsed = time.perf_counter() - start
assert results == ["a: done", "b: done", "c: done"]
assert elapsed < 0.35


参考实现：


In [ ]:
async def fetch_all(requests):
    tasks = [asyncio.create_task(fake_fetch(name, delay)) for name, delay in requests]
    return await asyncio.gather(*tasks)

sample_requests = [("a", 0.1), ("b", 0.2), ("c", 0.05)]
start = time.perf_counter()
results = await fetch_all(sample_requests)
elapsed = time.perf_counter() - start
print("input:", sample_requests)
print("output:", results)
print("elapsed_seconds:", round(elapsed, 3))

assert results == ["a: done", "b: done", "c: done"]
assert elapsed < 0.35


### 练习 B：并发执行但保持结果顺序

思路精髓：`gather` 返回结果的顺序由传入任务顺序决定，不由完成顺序决定。


In [ ]:
async def delayed_square(x):
    await asyncio.sleep(0.03 * (4 - x))
    return x * x

async def square_all_in_order(nums):
    # TODO
    pass

assert await square_all_in_order([1, 2, 3]) == [1, 4, 9]


参考实现：


In [ ]:
async def delayed_square(x):
    await asyncio.sleep(0.03 * (4 - x))
    return x * x

async def square_all_in_order(nums):
    tasks = [asyncio.create_task(delayed_square(x)) for x in nums]
    return await asyncio.gather(*tasks)

sample_nums = [1, 2, 3]
sample_output = await square_all_in_order(sample_nums)
print("input:", sample_nums)
print("output:", sample_output)

assert sample_output == [1, 4, 9]
